# 🌧️ Rainfall Prediction System - Interactive EDA & Machine Learning Notebook

This notebook contains the complete step-by-step code and live outputs for exploratory data analysis (EDA) and Logistic Regression classification.


### 📌 Step 1: Load Dataset & Inspect Data Quality
We load the original `Dataset/rainfall.csv` and clean extra whitespace from column names.


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Load the original dataset
df = pd.read_csv("Dataset/rainfall.csv")

# 2. Clean extra whitespaces from column names
df.columns = df.columns.str.strip()

print("Dataset Shape:", df.shape)
print("Clean Columns:", list(df.columns))
print("Duplicate Rows:", df.duplicated().sum())
df.head()


### 🔍 Step 2: Missing Values Analysis
Inspect missing values across all meteorological features.


In [1]:
missing_counts = df.isnull().sum()
missing_df = pd.DataFrame({
    'Feature': missing_counts.index,
    'Missing_Count': missing_counts.values,
    'Percentage (%)': (missing_counts.values / len(df)) * 100
}).sort_values(by='Missing_Count', ascending=False)

print(missing_df.to_string(index=False))

# Plot missing values count
plt.figure(figsize=(8, 4))
bars = plt.bar(range(len(missing_df['Feature'])), missing_df['Missing_Count'], color='#0284c7', edgecolor='#0369a1')
plt.title("Missing Values Count per Feature", fontsize=13, fontweight='bold')
plt.xlabel("Features", fontweight='bold')
plt.ylabel("Missing Count", fontweight='bold')
plt.xticks(range(len(missing_df['Feature'])), missing_df['Feature'], rotation=45, ha='right')
plt.ylim(0, max(max(missing_df['Missing_Count']) + 2, 5))
for b in bars:
    h = b.get_height()
    plt.annotate(f"{int(h)}", xy=(b.get_x() + b.get_width() / 2, h), xytext=(0, 3),
                 textcoords="offset points", ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()


### 🌧️ Step 3: Target Variable Distribution (Rainfall: Yes vs No)
Analyze class balance and occurrence frequency.


In [1]:
target_counts = df['rainfall'].str.strip().str.lower().value_counts()
print("Class Distribution:")
print(target_counts)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4.5))
colors = ["#38bdf8", "#fbbf24"]

# Bar Chart
ax1.bar(target_counts.index.str.upper(), target_counts.values, color=colors, edgecolor='#0284c7', width=0.5)
ax1.set_title("Rainfall Class Counts", fontweight='bold')
ax1.set_ylabel("Days", fontweight='bold')
for b in ax1.patches:
    h = b.get_height()
    ax1.annotate(f"{int(h)} ({h/len(df):.1%})", xy=(b.get_x() + b.get_width() / 2, h / 2),
                 ha='center', va='center', fontweight='bold', color='#0f172a')

# Donut Chart
ax2.pie(
    target_counts.values,
    labels=[f"Rain ({target_counts['yes']})", f"No Rain ({target_counts['no']})"],
    autopct='%1.1f%%',
    startangle=140,
    colors=colors,
    wedgeprops=dict(width=0.45, edgecolor="white", linewidth=2)
)
ax2.set_title("Rainfall Ratio", fontweight='bold')
plt.suptitle("Target Variable: Rainfall Occurrence (Yes vs No)", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


### 🌡️ Step 4: Temperature Distribution & Thermal Variations
Analyze ambient, maximum, and minimum temperature distributions.


In [1]:
mean_temp = df['temparature'].mean()
median_temp = df['temparature'].median()
print(f"Mean Temperature:   {mean_temp:.2f}°C")
print(f"Median Temperature: {median_temp:.2f}°C")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.5))

# Histogram with KDE
sns.histplot(df['temparature'], kde=True, ax=ax1, color="#0284c7", edgecolor="white", bins=20)
ax1.axvline(mean_temp, color="#dc2626", linestyle="--", linewidth=1.8, label=f"Mean: {mean_temp:.1f}°C")
ax1.axvline(median_temp, color="#16a34a", linestyle="-.", linewidth=1.8, label=f"Median: {median_temp:.1f}°C")
ax1.set_title("Temperature Histogram with KDE", fontweight='bold')
ax1.legend()

# Min, Avg, Max Boxplots
sns.boxplot(data=df[['mintemp', 'temparature', 'maxtemp']], ax=ax2, palette=["#38bdf8", "#0284c7", "#f97316"], width=0.5)
ax2.set_title("Temperature Variations (Min, Avg, Max)", fontweight='bold')
ax2.set_xticks(range(3))
ax2.set_xticklabels(["Min Temp", "Avg Temp", "Max Temp"], fontweight='bold')

plt.suptitle("Thermal Distribution Analysis", fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


### 🔥 Step 5: Correlation Heatmap
Compute Pearson correlation between weather attributes and rainfall.


In [1]:
df['rainfall_numeric'] = df['rainfall'].astype(str).str.strip().str.lower().map({'yes': 1, 'no': 0})
feature_cols = [
    'pressure', 'maxtemp', 'temparature', 'mintemp', 'dewpoint',
    'humidity', 'cloud', 'sunshine', 'winddirection', 'windspeed', 'rainfall_numeric'
]
corr_matrix = df[feature_cols].corr()

plt.figure(figsize=(9, 7))
sns.heatmap(
    corr_matrix,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    cbar_kws={'label': 'Pearson Correlation'},
    linewidths=0.5,
    linecolor="#e2e8f0"
)
plt.title("Correlation Heatmap: Meteorological Features vs Rainfall", fontsize=13, fontweight='bold', pad=12)
plt.xticks(rotation=45, ha='right', fontweight='bold')
plt.yticks(rotation=0, fontweight='bold')
plt.tight_layout()
plt.show()


### 🤖 Step 6: Logistic Regression Model Training & Confusion Matrix
Train model inside scikit-learn Pipeline (Imputer + Scaler + LogisticRegression) on 80:20 split.


In [1]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

input_features = [
    'pressure', 'maxtemp', 'temparature', 'mintemp', 'dewpoint',
    'humidity', 'cloud', 'sunshine', 'winddirection', 'windspeed'
]
X = df[input_features]
y = df['rainfall_numeric']

# 80:20 Stratified Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Pipeline
pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('classifier', LogisticRegression(random_state=42, max_iter=1000))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print("=" * 50)
print(f"Accuracy:  {accuracy_score(y_test, y_pred):.4f} ({accuracy_score(y_test, y_pred)*100:.2f}%)")
print(f"Precision: {precision_score(y_test, y_pred):.4f} ({precision_score(y_test, y_pred)*100:.2f}%)")
print(f"Recall:    {recall_score(y_test, y_pred):.4f} ({recall_score(y_test, y_pred)*100:.2f}%)")
print(f"F1 Score:  {f1_score(y_test, y_pred):.4f} ({f1_score(y_test, y_pred)*100:.2f}%)")
print("=" * 50)
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=['No Rain', 'Rain']))

cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    annot_kws={"size": 14, "weight": "bold"},
    xticklabels=["Predicted: No Rain", "Predicted: Rain"],
    yticklabels=["Actual: No Rain", "Actual: Rain"]
)
plt.title(f"Confusion Matrix (Accuracy: {accuracy_score(y_test, y_pred):.1%})", fontweight='bold', pad=12)
plt.tight_layout()
plt.show()
